# Part 4 - Competition CatBoost Pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

TARGET='addicted_label'
ID='id'

# Expects train_fe/test_fe from Part 3
FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES]
y=train_fe[TARGET]
X_test=test_fe[FEATURES]

CAT_FEATURES=X.select_dtypes(include=['object','category']).columns.tolist()
cat_idx=[X.columns.get_loc(c) for c in CAT_FEATURES]


## Training Configuration

In [ ]:
params={
    'loss_function':'Logloss',
    'eval_metric':'AUC',
    'iterations':5000,
    'learning_rate':0.02,
    'depth':8,
    'l2_leaf_reg':5,
    'subsample':0.8,
    'random_seed':42,
    'verbose':250
}

cv=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof=np.zeros(len(X))
test_pred=np.zeros(len(X_test))
scores=[]
feature_importance=pd.DataFrame(index=FEATURES)


## Cross Validation

In [ ]:
for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print("="*60)
    print("Fold",fold)

    model=CatBoostClassifier(**params)

    model.fit(
        X.iloc[tr],
        y.iloc[tr],
        cat_features=cat_idx,
        eval_set=(X.iloc[va],y.iloc[va]),
        use_best_model=True,
        early_stopping_rounds=300
    )

    val_pred=model.predict_proba(X.iloc[va])[:,1]
    oof[va]=val_pred

    fold_auc=roc_auc_score(y.iloc[va],val_pred)
    scores.append(fold_auc)

    print("Fold AUC:",round(fold_auc,6))

    test_pred+=model.predict_proba(X_test)[:,1]/cv.n_splits

    feature_importance[f'fold_{fold}']=model.get_feature_importance()

print("\nMean CV:",np.mean(scores))
print("Std CV :",np.std(scores))


## Overall ROC-AUC

In [ ]:
overall=roc_auc_score(y,oof)
print("Overall OOF ROC-AUC:",overall)


## Feature Importance

In [ ]:
feature_importance['mean']=feature_importance.mean(axis=1)
feature_importance=feature_importance.sort_values('mean',ascending=False)

display(feature_importance.head(20))

plt.figure(figsize=(8,8))
feature_importance['mean'].head(20).sort_values().plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()


## Submission

In [ ]:
submission=sample.copy()
submission[TARGET]=test_pred
submission.to_csv('submission_catboost.csv',index=False)

display(submission.head())

print('Saved: submission_catboost.csv')


## Notes

- This is a strong CatBoost baseline.
- Save the OOF predictions for later blending with LightGBM and XGBoost.
- In Part 5 we'll build a tuned LightGBM pipeline and prepare for ensembling.
